In [ ]:
%load_ext autoreload
%autoreload 2 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import scipy.stats as stats
import os
import sys
import matplotlib.pyplot as plt
pd.set_option('display.max_rows', 500)
import importlib
import os
import sys
#root_path = os.path.dirname(os.path.dirname(os.path.abspath(os.path.dirname('__file__'))))

root_path = os.path.dirname(os.path.abspath(os.path.dirname('__file__')))
sys.path.insert(0, root_path)
from env.parameters import P
from analysis_functions.data_preparation import cohort_type_adjustment
import dask.dataframe as dd
import pickle
import yaml
from analysis_functions.feature_engineering import (
keep_england_country_imd,
drop_unknown_country_imd,
imd_quantiles,
change_level_grouped_eth,
change_level_smoking,
change_level_alcohol,
find_normal_boundaries, 
find_skewed_boundaries,
diagnostic_plots,
plot_boxplot_and_hist,
outlier_analysis,
keep_england_country_imd,
change_level_grouped_eth,
change_level_smoking,
change_level_alcohol,
impute_nulls_mice,
create_age_bands
)

from analysis_functions.custom_transformers import (
Custom_Winsoriser,
bmi_categoriser,
fev1fvc_ratio_categoriser,
traffic_intensity_quantiles,
inverse_distance_quantiles,
CustomFrequencyBinner,
CustomWaistBinner,
MultiTransform,
MultiTransformList,
CustomBMICategoriser,
CustomFev1FvcRatioCategoriser,
CustomInverseDistanceCategoriser,
CustomTrafficIntensityCategoriser,
ColumnSelector,
CustomBinaryCategoriserAroundMean,
CustomBinaryCategoriserAroundMedian,
CustomBinaryCategoriserAroundDecile
)
from scipy.stats.mstats import winsorize
from fancyimpute import IterativeImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer
from scipy.stats import shapiro
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler
import warnings
from diffprivlib.utils import PrivacyLeakWarning
from sklearn.decomposition import PCA
import diffprivlib as dp
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
accuracy_score, confusion_matrix, 
classification_report, f1_score, 
roc_curve, roc_auc_score,
precision_recall_curve,
precision_score, recall_score, average_precision_score,balanced_accuracy_score, matthews_corrcoef)
import shap
from scipy.stats import chi2

from sklearn.neighbors import NearestNeighbors
from pipeline_functions import *
from custom_plots import *
from sklearn.neighbors import NearestNeighbors
from scipy.stats import fisher_exact, norm, chi2_contingency
from statsmodels.stats.contingency_tables import Table2x2

from pipeline_functions import *
from custom_plots import *
from epi_functions import *

from sklearn.neighbors import NearestNeighbors
from scipy.stats import fisher_exact, norm, chi2_contingency
from statsmodels.stats.contingency_tables import Table2x2
from collections import Counter

In [ ]:
cohort_path = P.cohorts_ukb_start_gphesonly_path

In [ ]:
df_in = pd.read_csv(f'''{P.cohorts_ukb_start_gphesonly_path}/analysis_csv/epi_analysis_ready_df_ukb_start_gphesonly.csv''')

In [ ]:
pickle_file = f'''{P.cohorts_ukb_start_gphesonly_path}/pickle/cols_dict_gphesonly_all.pickle'''

with open(pickle_file, 'rb') as f:
     cols_dict = pickle.load(f)

print(cols_dict.keys())

In [ ]:
print(df_in.shape)



In [ ]:
# step 1
df_in = cohort_type_adjustment(df_in, cols_dict)




In [ ]:
df_in["country_imd"].value_counts(dropna=False)

In [ ]:
df_in.shape

# Feature selection

# How many with less than 1 year of follow up?

In [ ]:
col_o = "flag_post_cohort_start_exac_ocs_y1"

In [ ]:
df_in[df_in["follow_up_asthma_pre_cohort_start"]<1].shape[0]

In [ ]:
df_in[df_in["follow_up_asthma_pre_cohort_start"]<1]["follow_up_asthma_pre_cohort_start"].describe()

In [ ]:
# Drop these
df = df_in[df_in["follow_up_asthma_pre_cohort_start"]>=1]

In [ ]:
# Any outcome on study start?

df[df['evdt_first_post_cohort_start_exac']==df['evdt_cohort_start']]

In [ ]:
print(df_in.shape)
print(df.shape)

In [ ]:
df_in.shape[0] - df.shape[0]

In [ ]:
df[col_o].value_counts()/df.shape[0]*100

In [ ]:
df.loc[:, "late_onset_asthma_40"] = df["age_asthma"].apply(lambda x: 1 if x >=40 else 0)

# Cardinal symptoms

In [ ]:
df.loc[:, "cardinal_symptoms"] =df[["wheeze_field", "shortness_breath_field", "chest_pain_field"]].any(axis=1).astype(int)

In [ ]:
df["cardinal_symptoms"].value_counts()

# How many deaths before the outcome (do not censor). Just do sensitivity without these later

In [ ]:
df[["evdt_cohort_start","evdt_first_post_cohort_start_exac", "dod"]].dtypes

In [ ]:
#condition = (df[col_o]==1) & (df['dod'].notnull()) & ((df['dod'] >= df['evdt_first_post_cohort_start_exac']) & (df['dod'] <= df['evdt_first_post_cohort_start_exac'] + pd.Timedelta(days=365)))
condition = (df['dod'].notnull()) & (df['dod'] > df['evdt_cohort_start']) & (df['dod'] <= df['evdt_cohort_start'] + pd.Timedelta(days=365))

df[condition].shape

In [ ]:
# any exac in these?
df[condition & df[col_o]==1][["evdt_cohort_start", "evdt_first_post_cohort_start_exac", "dod"]].shape

In [ ]:
df['tte_cohort_start_to_exac'].describe()

In [ ]:
# any deaths recorded before exac 1 event data? any deaths in non-exac within a year?

df[df['dod']<df['evdt_first_post_cohort_start_exac']]

# One hot encoded smoking and ethnicity

- We have not dropped any level here. Drop in modeling

In [ ]:
df = pd.get_dummies(df, columns=['eth_grouped_1b', 'desc_smoking_at_baseline'], drop_first=False)
dummy_columns = [col for col in df.columns if 'eth_grouped_1b_' in col or 'desc_smoking_at_baseline_' in col]
df[dummy_columns] = df[dummy_columns].astype(int)

In [ ]:
df['Smoker_current'] = df["desc_smoking_at_baseline_Current"].apply(lambda x: 1 if x ==1 else 0)

In [ ]:
np_random_seed = 7
dp_random_seed = 47

In [ ]:
# Covariates
cov_list= ['age_60+', 'sex_female', 
           'eth_non_white',
            'pheno_anxiety_pre_cohort_start',
           'bmi_30_imputed', 
           'pheno_ckd_pre_cohort_start',
           'pheno_copd_pre_cohort_start',
           'pheno_cvd_pre_cohort_start',
           'pheno_diabetes_pre_cohort_start',
           'pheno_ht_pre_cohort_start',
           'cardinal_symptoms',
            'flag_pre_cohort_start_exac_y1', 'flag_pre_cohort_start_meds_ocs_y1'] 



In [ ]:
# Covaraite rename
rename_dict = {
 'age_60+': "Age≥60",
 'sex_female': "Female sex",
 'eth_non_white': "Non white",
 'pheno_anxiety_pre_cohort_start': "Anxiety",
 'bmi_30_imputed': "BMI≥30",
 'pheno_ckd_pre_cohort_start': "CKD",
 'pheno_copd_pre_cohort_start' :"COPD",
 'pheno_cvd_pre_cohort_start': "CVD",
 'pheno_diabetes_pre_cohort_start': "Diabetes",
 'pheno_ht_pre_cohort_start' : "Hypertension",
 'cardinal_symptoms': "Cardinal symptomps",
 'flag_pre_cohort_start_exac_y1':  "Pre exacerbation",
 'flag_pre_cohort_start_meds_ocs_y1': "Pre OCS"
}


In [ ]:
df = df.rename(columns=rename_dict)

In [ ]:
cov_list = rename_dict.values()

# Propensity score

In [ ]:
dummy_columns

In [ ]:

dict_categories = {
    'eth_grouped_1b': ['White', 'Other', 'Mixed', 'Asian', 'Black'],
}

In [ ]:

df_ohe= df

In [ ]:
from sklearn.preprocessing import MinMaxScaler
numerical_vars = ['age_cohort_start', 'follow_up_asthma_pre_cohort_start']

# MinMaxScaler
scaler = MinMaxScaler()
df['age_cohort_start_min_max'] = scaler.fit_transform(df[["age_cohort_start"]])
df['follow_up_min_max'] = scaler.fit_transform(df[["follow_up_asthma_pre_cohort_start"]])


In [ ]:
covariates = ["age_cohort_start_min_max", 
            "Pre_baseline_exacerbation",
            "Pre_baseline_OCS",
            "COPD", 
            'Sex_female',
            'eth_non_white',
             "follow_up_min_max"
            'Smoker_current']
rename_dict_covs = {
 'age_cohort_start_min_max': "Age",
"Pre_baseline_exacerbation": "Pre exacerbation",
"Pre_baseline_OCS": "Pre OCS",
"COPD": "COPD",
'Sex_female': 'Female sex',
'eth_non_white': "Non white",
'Smoker_current': "Current smoker",
'follow_up_min_max': "Asthma follow-up"
}

In [ ]:
df_ohe = df_ohe.rename(columns=rename_dict_covs)

In [ ]:
cov_list_2 = rename_dict_covs.values()
cov_list_2

In [ ]:
np_model = LogisticRegression(max_iter=500, random_state=np_random_seed)
np_model.fit(df_ohe[cov_list_2], df_ohe[col_o])

In [ ]:
df_ohe.loc[:, 'propensity_score'] = np_model.predict_proba(df_ohe[cov_list_2])[:, 1]

In [ ]:
knn_count = 1

# KNN based on propensity score

In [ ]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors
import numpy as np

def knn_matching_updated(df_ohe, col_o, n_neighbours=4, caliper=None):
    """ Performs 1:N nearest neighbors matching on the dataset based on propensity scores without replacement.

    df_ohe (DataFrame): The dataframe with propensity scores and other variables.
    col_o (str): The column name indicating treated units (1 for treated, 0 for control).
    n_neighbours (int): Number of control matches per treated unit (default is 4).
    caliper (float): The maximum allowed difference in propensity scores for matching (default is None).
    
    Returns:
    df_knn_matched (DataFrame): A dataframe containing matched treated and control units.
    duplicates (int): The number of duplicated control matches (expected to be zero without replacement).
    """
    # Split the dataset into cases and controls
    case = df_ohe[df_ohe[col_o] == 1].copy()  
    control = df_ohe[df_ohe[col_o] == 0].copy()
    # A list to hold matched control indices
    matched_control_indices = []
    for i in range(len(case)):
        # Get the propensity score
        treated_unit_score = case.iloc[[i]][['propensity_score']]
        # Fit KNN
        # L1 norm for absolute difference
        knn = NearestNeighbors(n_neighbors=n_neighbours, metric='manhattan')  
        knn.fit(control[['propensity_score']])
        # Find the nearest neighbors in the control group 
        distances, indices = knn.kneighbors(treated_unit_score)
        selected_controls = control.iloc[indices.flatten()]
        # Apply  caliper 
        if caliper is not None:
            selected_controls = selected_controls[np.abs(selected_controls['propensity_score'] - treated_unit_score.values[0][0]) <= caliper]

        matched_control_indices.extend(selected_controls.index)
        # Remove the selected controls from the control group to avoid replacement
        control = control.drop(selected_controls.index)
    matched_controls = df_ohe.loc[matched_control_indices]
    df_knn_matched = pd.concat([case, matched_controls])
    df_knn_matched.reset_index(drop=True, inplace=True)
    duplicates = pd.Series(matched_control_indices).duplicated().sum()
    print(f"Number of duplicated matches: {duplicates}")
    return df_knn_matched, duplicates

In [ ]:
df_knn_matched, _ = knn_matching_updated(df_ohe, col_o, n_neighbours=knn_count, caliper=0.1)

In [ ]:
df_knn_matched.shape

In [ ]:
df_ohe[col_o].value_counts()

In [ ]:
df_knn_matched[col_o].value_counts()

In [ ]:
df_ohe[col_o].value_counts()/df_ohe.shape[0]*100

In [ ]:
df_knn_matched[col_o].value_counts()/df_knn_matched.shape[0]*100

In [ ]:
df_knn_matched[df_knn_matched["eid"].duplicated()].shape

In [ ]:
def calculate_standardised_mean_difference(df, covariates, treatment_col):
    """
    Calculate the standardized mean differences (SMDs) for the covariates.
    
    Parameters:
    - df: DataFrame containing the matched data.
    - covariates: List of covariate column names.
    - treatment_col: Name of the treatment column.
    
    Returns:
    - A Series containing the SMDs for each covariate.
    """
    means_treated = df[df[treatment_col] == 1][covariates].mean()
    means_control = df[df[treatment_col] == 0][covariates].mean()
    std_pooled = np.sqrt((df[covariates].var().values + df[covariates].var().values) / 2)
    smd = np.abs(means_treated - means_control) / std_pooled
    return smd


In [ ]:

smds = calculate_standardised_mean_difference(df_knn_matched, cov_list_2, col_o)
print(smds)

In [ ]:
smds_before_matching = calculate_standardised_mean_difference(df_ohe, cov_list_2, col_o)
print(smds_before_matching)

In [ ]:
cov_list_2

In [ ]:
smds_before_matching

In [ ]:
smds

In [ ]:
import matplotlib.pyplot as plt

def plot_smd_comparison_2(smds_before_matching, 
                        smds_after_matching, 
                        covariates, figsize=(6, 4),
                        title='SMD Before and After Matching',
                         output_path='smd_comparison.tiff'):
    """
    Plots the standardized mean differences (SMD) before and after matching for covariates.
    
    Parameters:
    smds_before_matching (list or array): SMD values before matching.
    smds_after_matching (list or array): SMD values after matching.
    covariates (list): List of covariate names.
    figsize (tuple): Size of the figure (default is (6, 4)).
    
    Returns:
    None
    """
    plt.figure(figsize=figsize)

    plt.plot(smds_before_matching, covariates, linestyle='--', marker='o', 
             label='Before Matching', color='black', markersize=4, markerfacecolor='white')
    plt.plot(smds_after_matching, covariates, linestyle='-', marker='s', 
             label='After Matching', color='black', markersize=5)
    plt.axvline(x=0.1, color='lightgrey', linestyle='--', linewidth=1.2)
    plt.xlabel('Standard Mean Difference', fontsize=9)
    plt.ylabel('Baseline covariates', fontsize=10)
    plt.title(title, fontsize=9)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize=9, frameon=False)
    plt.xticks([0, 0.1, 0.2, 0.3, 0.4], fontsize=9)
    plt.yticks(fontsize=9)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(output_path, format='tiff', dpi=300)
    plt.show()

In [ ]:
plot_smd_comparison_2(smds_before_matching = smds_before_matching, 
                    smds_after_matching= smds, 
                      title="",
                    covariates=cov_list_2, figsize=(5.2, 4), output_path = "comparison.tiff")

# Matched Odds Ratio based on NP propensity score

In [ ]:
matched_ids = df_knn_matched["eid"].unique()
df_matched = df[df["eid"].isin(matched_ids)]

In [ ]:
df_matched[col_o].value_counts()

In [ ]:
df_matched[col_o].value_counts()/df_matched.shape[0]*100

In [ ]:
for item in cov_list:
    print(f'{df_matched[item].value_counts()}')

In [ ]:
for item in cov_list:
    print(f'{df_matched[item].value_counts()/df_matched.shape[0]*100}')

In [ ]:
cov_list


In [ ]:
df_matched_heatmap = df_matched.copy()
df_matched_heatmap['*Age≥60'] = df_matched_heatmap['Age≥60']
df_matched_heatmap['*Female sex'] = df_matched_heatmap['Female sex']
df_matched_heatmap['*Non white'] = df_matched_heatmap['Non white']
df_matched_heatmap['*COPD'] = df_matched_heatmap['COPD']
df_matched_heatmap['*Pre exacerbation'] = df_matched_heatmap['Pre exacerbation']
df_matched_heatmap['*Pre OCS'] = df_matched_heatmap['Pre OCS']

In [ ]:
cov_list_heatmap= ['*Age≥60', '*Female sex', '*Non white',
                   'Anxiety', 'BMI≥30', 'CKD',
                   '*COPD', 'CVD', 'Diabetes', 'Hypertension',
                   'Cardinal symptomps', '*Pre exacerbation', '*Pre OCS']

In [ ]:
df_prevalence, df_proportion = heatmap_df_maker(df_matched_heatmap, col_o, cov_list_heatmap)
# Plot heatmap for Prevalence
plot_heatmap(df_prevalence, col_o, 'Prevalence (%)', cmap='Blues')

In [ ]:


# Plot heatmap for Proportion
plot_heatmap(df_proportion, col_o, 'Proportion (%)', cmap='Oranges')

In [ ]:
df_matched[col_o].value_counts()

In [ ]:
import numpy as np
import scipy.stats as stats
df_prevalence['Prevalence in cases (%)'] = df_prevalence['Prevalence in cases (%)'].round(2)
df_prevalence['Prevalence in controls (%)'] = df_prevalence['Prevalence in controls (%)'].round(2)
# Prevalence Ratio (PR) and Prevalence Difference
df_prevalence['Prevalence Ratio (PR)'] = df_prevalence['Prevalence in cases (%)'] / df_prevalence['Prevalence in controls (%)']
df_prevalence['Prevalence Ratio (PR)'] = df_prevalence['Prevalence Ratio (PR)'].round(2)
df_prevalence['Prevalence difference'] = df_prevalence['Prevalence in cases (%)'] - df_prevalence['Prevalence in controls (%)']
df_prevalence['CI Lower'] = np.nan  # Placeholder for lower CI
df_prevalence['CI Upper'] = np.nan  # Placeholder for upper CI
df_prevalence['p-value'] = np.nan   # Placeholder for p-value
n1 = (df_matched[col_o] == 1).sum()  # Total number of cases
n2 = (df_matched[col_o] == 0).sum()  # Total number of controls
print(n1)
print(n2)
for index, row in df_prevalence.iterrows():
    try:
        pr = row['Prevalence Ratio (PR)']
        log_pr = np.log(pr) if pr > 0 else 0  # Avoid log(0)
        se_log_pr = np.sqrt((1 / n1) + (1 / n2))
        ci_log_pr_lower = log_pr - 1.96 * se_log_pr
        ci_log_pr_upper = log_pr + 1.96 * se_log_pr
        ci_lower = np.exp(ci_log_pr_lower) if se_log_pr > 0 else np.nan
        ci_upper = np.exp(ci_log_pr_upper) if se_log_pr > 0 else np.nan
        z_score = log_pr / se_log_pr if se_log_pr > 0 else 0
        p_value = 2 * (1 - stats.norm.cdf(abs(z_score))) if se_log_pr > 0 else np.nan
        df_prevalence.at[index, 'CI Lower'] = round(ci_lower, 2)
        df_prevalence.at[index, 'CI Upper'] = round(ci_upper, 2)
        df_prevalence.at[index, 'p-value'] = round(p_value, 4)
    except Exception as e:
        print(f"Error at index {index}: {e}")

In [ ]:
df_prevalence

In [ ]:
df_prevalence.to_csv('7_matched_0747_df_prevalence.csv', index=False)  

In [ ]:
for item in df_prevalence['Prevalence in controls (%)']:
    print(item)

In [ ]:
for item in cov_list_heatmap:
    print(item)
    print(df_matched_heatmap[df_matched_heatmap[item]==1][col_o].value_counts())

In [ ]:
for item in cov_list_heatmap:
    print(item)
    print(df_matched_heatmap[(df_matched_heatmap[item]==1)&(df_matched_heatmap[col_o]==1)].shape[0])

In [ ]:
for item in cov_list_heatmap:
    n = df_matched_heatmap[(df_matched_heatmap[item]==1)&(df_matched_heatmap[col_o]==1)].shape[0]
    denom = df_matched_heatmap[df_matched_heatmap[col_o]==1].shape[0]
    print(f'''{n} ({round(n/denom*100, 2)})''') 

In [ ]:
for item in cov_list_heatmap:
    n = df_matched_heatmap[(df_matched_heatmap[item]==1)&(df_matched_heatmap[col_o]==0)].shape[0]
    denom = df_matched_heatmap[df_matched_heatmap[col_o]==0].shape[0]
    print(f'''{n} ({round(n/denom*100, 2)})''') 

In [ ]:
cov_list

In [ ]:
# Separate outcome
X = df_matched[cov_list]

y = df_matched[[col_o]].values.flatten()


In [ ]:
# Baseline model without differential privacy
np_lr_model = LogisticRegression(random_state=np_random_seed)
np_lr_model.fit(X, y)

In [ ]:
np_coeff_df = make_OR_from_LR_output(np_lr_model, X)
np_coeff_df

In [ ]:
np_results_df = calculate_rr_or_ci_pvalue(np_lr_model, X, y)
np_results_df

In [ ]:
np_results_df.to_csv('7_matched_0747_df_np_results.csv', index=False)  

In [ ]:
def plot_forest_log_scale_method_22(df, type="RR", figsize=(10, 6), log_scale=True, xtick_fontsize=10, show_p_value=True):
    """Forest plot of a single model in log scale with ORs/RRs, CIs, and p-values listed on the right."""
    sns.set(style="white")

    if type == "OR":
        ratio_text = 'or'
        ci_lower_text = 'or_ci_lower'
        ci_upper_text = 'or_ci_upper'
        p_value_text = 'or_p_value'
        title_text = "Odds"
    else:
        ratio_text = 'rr'
        ci_lower_text = 'rr_ci_lower'
        ci_upper_text = 'rr_ci_upper'
        p_value_text = 'rr_p_value'
        title_text = "Risk"

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(df[ratio_text], df['Covariate'], color='none')
    ax.invert_yaxis()
    for i, row in df.iterrows():
        fmt = 's'
        mfc_marker = 'white' if row[p_value_text] >= 0.05 else 'darkblue'
        ax.errorbar(row[ratio_text], row['Covariate'],
                    xerr=[[row[ratio_text] - row[ci_lower_text]], [row[ci_upper_text] - row[ratio_text]]],
                    fmt=fmt, mfc=mfc_marker, color='darkblue' if row[p_value_text] < 0.05 else 'darkblue', ecolor='royalblue', capsize=0, markersize=4)

    ax.axvline(x=1, color='gray', linestyle=':', linewidth=1)
    if log_scale:
        ax.set_xscale('log')
    ax.set_xlabel(f'1:{knn_count} Matched OR {"(log scale)" if log_scale else ""}')
    ax.set_title(f'Forest Plot of {title_text} Ratios with 95% Confidence Intervals')
    ax.grid(True, linestyle='--', alpha=0.2)
    ax.tick_params(axis='y', which='major', labelsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(True)  
    ax.yaxis.set_ticks_position('none') 

    # Add a column with OR/RR, CI, and p-value text outside the plot area
    xlim = ax.get_xlim()    
    for i, row in df.iterrows():
        ratio_value = f"{row[ratio_text]:.2f}"
        ci_text = f"({row[ci_lower_text]:.2f}, {row[ci_upper_text]:.2f})"
        if row[p_value_text] < 0.001:
            p_value_text_display = "p < 0.001"
        elif row[p_value_text] < 0.01:
            p_value_text_display = "p < 0.01"
        elif row[p_value_text] < 0.05:
            p_value_text_display = "p < 0.05"
        else:
            p_value_text_display = f"p= {row[p_value_text]:.2g}"
        y_value = row['Covariate']
        ax_text = f"{ratio_value} {ci_text} {p_value_text_display}"
        if not show_p_value:
            ax_text = f"{ratio_value} {ci_text}"
        x_text_position = 6.1

        ax.text(x_text_position, y_value, 
                ax_text, 
                ha='left', va='center', fontsize=9, color='black')
       
    ax.set_xlim(left=0, right=6)
    ax.xaxis.set_tick_params(labelsize=xtick_fontsize)
    ax.margins(y=0.1) 
    plt.tight_layout()
    plt.show()

In [ ]:
plot_forest_log_scale_method_22(np_results_df, type="OR", figsize=(5, 5), log_scale= False, show_p_value=False)

In [ ]:
nrd2 = np_results_df.copy()
pl = ["Age≥60", "Female sexl", "Non white", "COPD", "Pre exacerbation", "Pre OCS"]
for item in pl:
    nrd2.loc[nrd2["Covariate"]==item, "Covariate"]= f'*{item}'

In [ ]:
plot_forest_log_scale_method_22(nrd2, type="OR", figsize=(5, 5), log_scale= False, show_p_value=False)

In [ ]:
from sklearn.calibration import calibration_curve

probs_model = np_lr_model.predict_proba(X)[:, 1]

# Calibration curve
fraction_of_positives, mean_predicted_value = calibration_curve(y, probs_model, n_bins=20)
plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="Logistic regression")
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.title('Calibration plot')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(4, 4))
plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="Non-private", color='darkblue',alpha=0.5)
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated", alpha=0.3)
plt.xlabel('Mean predicted probability', fontsize=10)
plt.ylabel('Fraction of positives', fontsize=10)
plt.xticks(fontsize=9)  
plt.yticks(fontsize=9)
plt.title(f'')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=12)
plt.grid(True, linestyle='--', alpha=0.2)  # Adjust alpha for transparency
plt.show()

In [ ]:
def calibrated_plot_all_epsilons(model_dict, X, y, matched=False, reduced_matched = False, fig_size=(6, 4)):
    
    probs_model1 = model_dict.get("model")[0].predict_proba(X)[:, 1]  

    for i, e in enumerate(model_dict.get("epsilon")[1:]):
        #print(e)
        probs_model2 = model_dict.get("model")[i+1].predict_proba(X)[:, 1]

        fraction_of_positives1, mean_predicted_value1 = calibration_curve(y, probs_model1, n_bins=10)
        fraction_of_positives2, mean_predicted_value2 = calibration_curve(y, probs_model2, n_bins=10)
        
        color = 'darkblue' if matched else 'black'
        if reduced_matched:
            color = 'darkolivegreen'

        plt.figure(figsize=fig_size)
        plt.plot(mean_predicted_value1, fraction_of_positives1, "s-", label="Non-private", color=color,alpha=0.5)
        plt.plot(mean_predicted_value2, fraction_of_positives2, "o-", label="Differentially private", color=color,alpha=0.3)
        plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated", alpha=0.3)
        plt.xlabel('Mean predicted probability', fontsize=10)
        plt.ylabel('Fraction of positives', fontsize=10)
        plt.xticks(fontsize=9)  
        plt.yticks(fontsize=9)
        plt.title(f'Calibration Plot Comparison of models, non-private vs differentially private (epsilon={e})')
        plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.2) 
        plt.show()

In [ ]:
from sklearn.metrics import brier_score_loss
probs_model = np_lr_model.predict_proba(X)[:, 1]
# Brier score
brier_score = brier_score_loss(y, probs_model)
print(f'Brier score: {brier_score}')

In [ ]:
from sklearn.metrics import roc_auc_score
# ROC AUC
roc = roc_auc_score(y, probs_model)
print(f'ROC AUC: {roc}')

In [ ]:

fpr, tpr, thresholds = roc_curve(y, probs_model)  # False Positive Rate, True Positive Rate
auc_score = roc_auc_score(y, probs_model)
# 4. Plot ROC Curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f'ROC curve (AUC = {auc_score:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')  # Diagonal line for random guessing
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.grid(True)
plt.show()

# SHAP

In [ ]:
shap.initjs()

In [ ]:
df_temp = pd.DataFrame(X, columns=cov_list)

df_temp = df_temp.apply(pd.to_numeric, errors='coerce')
df_temp = df_temp.astype('float64')

explainer = shap.LinearExplainer(np_lr_model, df_temp)

shap_values = explainer.shap_values(df_temp)

shap_explainer = explainer(df_temp)


In [ ]:
shap.plots.beeswarm(shap_explainer, max_display=len(cov_list), log_scale=True,  show=False)

plt.gcf().suptitle('Non-private')
plt.savefig('shap_matched_np.tiff', format='tiff', dpi=300, bbox_inches='tight')

plt.show()


# Data norm

In [ ]:
def row_to_tuple(row):
    return tuple(row)
combination_counts = Counter(df[cov_list].apply(row_to_tuple, axis=1))
comb_df = pd.DataFrame(list(combination_counts.items()), columns=['Combination', 'Count'])
comb_df = comb_df.sort_values('Count', ascending=False)
total_rows = len(df)
comb_df['Percentage'] = comb_df['Count'] / total_rows * 100


In [ ]:
print("\nSummary:")
print(f"Total number of rows: {total_rows}")
print(f"Number of unique combinations: {len(comb_df)}")
print(f"Most common combination occurs {comb_df['Count'].max()} times ({comb_df['Percentage'].max():.2f}%)")
print(f"Least common combination occurs {comb_df['Count'].min()} times ({comb_df['Percentage'].min():.2f}%)")
rare_combinations = comb_df[comb_df['Percentage'] < 1]
print(f"\nNumber of rare combinations (less than 1%): {len(rare_combinations)}")
#print(rare_combinations)

In [ ]:
len(cov_list)

In [ ]:
data_norm = np.sqrt(len(cov_list))
data_norm

# Differential privacy

In [ ]:
epsilons = [0.01, 0.025,0.05, 0.1, 0.25, 0.693, 1, 1.098, 2,5, 10]

In [ ]:
results_df_2, coeff_df_2,  model_dict= make_dp_or_rr_all_covariate_all_outputs(X, y, 
                                                                               epsilons=epsilons, 
                                                                               data_norm=data_norm, 
                                                                               random_state=dp_random_seed)

In [ ]:
# AIC BIC
aic_bic_results = {
        'epsilon': [x for x in model_dict.get('epsilon')],
        'AIC': [],
        'BIC': []
    }
log_likelihood_np = calculate_log_likelihood(model_dict.get('model')[0], X, y)
aic_np, bic_np = aic_bic(log_likelihood_np, X.shape[1], X.shape[0])
    
#aic_bic_results['epsilon'].append('None')
aic_bic_results['AIC'].append(aic_np)
aic_bic_results['BIC'].append(bic_np)
for e in reversed(epsilons):
    index = next(i for i, item in enumerate(model_dict['epsilon']) if item == e)
    dp_lr_model_temp = model_dict['model'][index]
    log_likelihood_dp = calculate_log_likelihood(dp_lr_model_temp, X, y)
    aic_dp, bic_dp = aic_bic(log_likelihood_dp, X.shape[1], X.shape[0])
    aic_bic_results['AIC'].append(aic_dp)
    aic_bic_results['BIC'].append(bic_dp)

aic_bic_df = pd.DataFrame(aic_bic_results)

In [ ]:
plot_aic_bic(aic_bic_df, matched=True)

In [ ]:
bland_altman_plot_all_epsilons(model_dict, X, matched=True)

In [ ]:
calibrated_plot_all_epsilons(model_dict, X, y, matched=True, fig_size=(4, 4))

In [ ]:
roc_curve_all_epsilons(model_dict, X, y, matched=True, fig_size=(4, 4))

In [ ]:
from sklearn.metrics import log_loss
import numpy as np
import matplotlib.pyplot as plt

# Function to calculate weighted log loss
def get_class_weights(y):
    n_positives = np.sum(y)
    n_negatives = len(y) - n_positives
    weight_positive = len(y) / (2 * n_positives)
    weight_negative = len(y) / (2 * n_negatives)
    
    return weight_positive, weight_negative

weight_positive, weight_negative = get_class_weights(y)

sample_weights = np.array([weight_positive if label == 1 else weight_negative for label in y])
probs_model1 = model_dict.get("model")[0].predict_proba(X)[:, 1]  # Probabilities for the positive class
log_losses = []
log_loss_value = log_loss(y, probs_model1, sample_weight=sample_weights)
log_losses.append(log_loss_value)
model_names = [f'None']

for i, e in enumerate(model_dict.get("epsilon")[1:]):
    probs_model2 = model_dict.get("model")[i + 1].predict_proba(X)[:, 1]
    log_loss_value = log_loss(y, probs_model2, sample_weight=sample_weights)
    log_losses.append(log_loss_value)
    model_names.append(f'{e}')


In [ ]:
# All metrics
pipe_accuracy = []
pipe_f1 = []
pipe_roc= [] 
pipe_precision = []
pipe_recall = []
pipe_ap = []
pipe_balanced_accuracy = []
pipe_mcc = []
pipe_brier = []
pipe_logloss = []
pipe_weighted_logloss = []
y_pred_full = model_dict.get("model")[0].predict_proba(X)[:, 1] 
roc_auc_score_full = roc_auc_score(y, y_pred_full)
ap_full = average_precision_score(y, y_pred_full)
brier_score_full = brier_score_loss(y, y_pred_full)
log_loss_full = log_loss(y, y_pred_full)
weighted_log_loss_full = log_loss(y, y_pred_full, sample_weight=sample_weights)

for i, e in enumerate(model_dict.get("epsilon")[1:]):
    y_pred = model_dict.get("model")[i].predict_proba(X)[:, 1] 
    pipe_roc.append(roc_auc_score(y, y_pred))
    pipe_ap.append(average_precision_score(y, y_pred))
    pipe_brier.append(brier_score_loss(y, y_pred))
    pipe_logloss.append(log_loss(y, y_pred))
    pipe_weighted_logloss.append(log_loss(y, y_pred, sample_weight=sample_weights))

In [ ]:
def plot_df_2_new(epsilons, pipe_metric, metric_full, title, y_label, ylim_low=0, ylim_high=1, log_scale=True, save_as_tiff=""):
    plt.figure(figsize=(6,4))
    if log_scale:
        plt.semilogx(epsilons, pipe_metric, label="Differentially private", zorder=10, color='darkblue', 
                 marker='.', markersize=7, linestyle='-')
    else:
        plt.plot(epsilons, pipe_metric, label="Differentially private", zorder=10, color='black', 
                 marker='.', markersize=7, linestyle='-')
        
    plt.plot(epsilons, np.ones_like(epsilons) * metric_full, dashes=[2,2], label=f'Baseline {y_label}: {round(metric_full, 2)}', 
             zorder=5, color='gray', linestyle='--')
    plt.title(title)
    plt.xlabel("Epsilon in log scale" if log_scale else "Epsilon")
    plt.ylabel(y_label)
    plt.ylim(ylim_low, ylim_high)
    plt.xlim(epsilons[0], epsilons[-1])
    plt.legend(loc=2)
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tick_params(axis='both', which='major', labelsize=10)
    plt.tick_params(axis='both', which='minor', labelsize=8)
    plt.tight_layout()
    if save_as_tiff != "":
        plt.savefig(f'''{save_as_tiff}.tiff''', format='tiff', dpi=300)
    plt.show()

In [ ]:
# ROC AUC
plot_df_2_new(epsilons[::-1], pipe_roc, roc_auc_score_full, title="", y_label="ROC AUC", save_as_tiff="matched_roc")


In [ ]:
# ROC AUC
plot_df_2_new(epsilons[::-1], pipe_ap, ap_full, title="", y_label="AP")

In [ ]:
plot_df_2_new(epsilons[::-1], pipe_brier, brier_score_full, 
              title="", y_label="Brier score",
             ylim_low=0, ylim_high=0.5, save_as_tiff="matched_brier")

In [ ]:
plot_df_2_new(epsilons[::-1], pipe_logloss, log_loss_full, 
              title="Differentially private Log-Loss score", y_label="Log-loss",
             ylim_low=0, ylim_high=18)

In [ ]:
plot_df_2_new(epsilons[::-1], pipe_weighted_logloss, weighted_log_loss_full, 
              title="Differentially private wighted log-loss score", y_label="Weighted Log-loss",
             ylim_low=0, ylim_high=18)

In [ ]:
explainer = shap.LinearExplainer(model_dict.get("model")[0], df_temp)

shap_values = explainer.shap_values(df_temp)

shap_values= explainer(df_temp)
shap.plots.beeswarm(shap_values, max_display=len(cov_list), log_scale=True, show=False)
plt.gcf().suptitle('Non-private')

plt.show()

In [ ]:
for i, e in enumerate(model_dict.get("epsilon")[1:]):
    explainer = shap.LinearExplainer(model_dict.get("model")[i], df_temp)

    shap_values = explainer.shap_values(df_temp)

    shap_values= explainer(df_temp)
    shap.plots.beeswarm(shap_values, max_display=len(cov_list), log_scale=True, show=False)
    plt.gcf().suptitle(f'Differentially private, epsilon= {e}')

    plt.show()

# Non logarithmic SHAP


In [ ]:
explainer = shap.LinearExplainer(model_dict.get("model")[0], df_temp)

shap_values = explainer.shap_values(df_temp)

shap_values= explainer(df_temp)
shap.plots.beeswarm(shap_values, max_display=len(cov_list), log_scale=False, show=False)
plt.gcf().suptitle('Non-private')

plt.show()

In [ ]:

for i, e in enumerate(model_dict.get("epsilon")[1:]):
    explainer = shap.LinearExplainer(model_dict.get("model")[i], df_temp)

    shap_values = explainer.shap_values(df_temp)

    shap_values= explainer(df_temp)
    shap.plots.beeswarm(shap_values, max_display=len(cov_list), log_scale=False, show=False)
    plt.gcf().suptitle(f'Differentially private, epsilon= {e}')

    plt.show()

# Differentialy private forest plots

In [ ]:
results_df_2.to_csv('7_matched_0747_df_differentially_private_outcomes.csv', index=False)  

In [ ]:
# BMI
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("bmi_30_imputed")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                
                               title = "",
                               covariate_name_xlabel= "BMI≥30", 
                               type="OR",
                               logscale=False,
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values= True, 
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0.1,
                               xlim_max=3.1,
                               text_x_position=3.2
                                
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("age_60+")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                    
                               title = "",
                               covariate_name_xlabel= "Age≥60",
                               type="OR",
                               logscale=False,
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=-0.1,
                               xlim_max=2.9,
                               text_x_position=3
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("sex_female")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                   
                               title = "",
                               covariate_name_xlabel= "Female sex",
                               type="OR",
                               logscale=False,
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=-0.1,
                               xlim_max=2.9,
                               text_x_position=3
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_anxiety_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                  
                               title = "",
                               covariate_name_xlabel= "Anxiety",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0.7,
                               xlim_max=3.7,
                               text_x_position=3.8
                          
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_ckd_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5,3.5),
                
                               title = "",
                               covariate_name_xlabel= "CKD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0.7,
                               xlim_max=3.7,
                               text_x_position=3.8
                          
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_ckd_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5,3.5),
                
                               title = "",
                               covariate_name_xlabel= "CKD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0.7,
                               xlim_max=12,
                               text_x_position=12.1

                          
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_ckd_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5,3.5),
                
                               title = "",
                               covariate_name_xlabel= "CKD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0,
                               xlim_max=4,
                               text_x_position=4.1, save_as_tiff="dp_matched_ckd"
                          
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_ckd_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5,3.5),
                
                               title = "",
                               covariate_name_xlabel= "CKD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0.5,
                               xlim_max=8,
                               text_x_position=8.1
                          
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_copd_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5,3.5),
                          
                               title = "",
                               covariate_name_xlabel= "COPD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0.5,
                               xlim_max=3.5,
                               text_x_position=3.6
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_cvd_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
               
                               title = "",
                               covariate_name_xlabel= "CVD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0.5,
                               xlim_max=3.5,
                               text_x_position=3.6
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_cvd_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
               
                               title = "",
                               covariate_name_xlabel= "CVD",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0.5,
                               xlim_max=8.0,
                               text_x_position=8.1
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_diabetes_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                    
                               title = "",
                               covariate_name_xlabel= "Diabetes",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0.3,
                               xlim_max=3.3,
                               text_x_position=3.4
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_diabetes_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                    
                               title = "",
                               covariate_name_xlabel= "Diabetes",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0.5,
                               xlim_max=8,
                               text_x_position=8.1
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("pheno_ht_pre_cohort_start")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
              
                               title = "",
                               covariate_name_xlabel= "Hypertension",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0.7,
                               xlim_max=3.7,
                               text_x_position=3.8
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("cardinal_symptoms")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                          
                               title = "",
                               covariate_name_xlabel= "Cardinal symptoms",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR\n",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=1.8,
                               xlim_max=4.8,
                               text_x_position=4.9
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("eth_non_white")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                           
                               title = "",
                               covariate_name_xlabel= "Non-white ethnicity",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR\n",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0.6,
                               xlim_max=3.6,
                               text_x_position=3.7
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("flag_pre_cohort_start_exac_y1")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                          
                               title = "",
                               covariate_name_xlabel= "1-year exacerbation\nclinical",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR\n",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=-0.1,
                               xlim_max=2.9,
                               text_x_position=3
                          )

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("flag_pre_cohort_start_meds_ocs_y1")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                       
                               title = "",
                               covariate_name_xlabel= "1-year OCS prescription",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=-0.1,
                               xlim_max=2.9,
                               text_x_position=3
                          )

In [ ]:
results_df_2[results_df_2['covariate']==rename_dict.get("flag_pre_cohort_start_meds_ocs_y1")]

In [ ]:
plot_or_rr_vertical_version_44(results_df_2[results_df_2['covariate']==rename_dict.get("flag_pre_cohort_start_meds_ocs_y1")],
                               text_offset=0.1, 
                               figsize=(3.5, 3.5),
                       
                               title = "",
                               covariate_name_xlabel= "1-year OCS prescription",
                               type="OR",
                               logscale=False, 
                               implausible_high_value= 20,
                               implausibly_high_text = "∞ (>20)",
                               show_all_values=True,
                               show_p_value=False, 
                                custom_x_label = f"1:{knn_count} Matched OR",
                                   x_size = 10, 
                                   y_size = 10, 
                                  adjustment = "matched",
                                
                               xlim_min=0,
                               xlim_max=4,
                               text_x_position=4.1, save_as_tiff="dp_matched_ocs"
                          )

In [ ]:
condition = (df['dod'].notnull()) & (df['dod'] > df['evdt_cohort_start']) & (df['dod'] <= df['evdt_cohort_start'] + pd.Timedelta(days=365))

df[condition].shape

In [ ]:
df[condition & df[col_o]==1][["evdt_cohort_start", "evdt_first_post_cohort_start_exac", "dod"]].shape

In [ ]:
df_knn_matched[(df_knn_matched['dod'].notnull()) & (df_knn_matched['dod'] > df_knn_matched['evdt_cohort_start']) & (df_knn_matched['dod'] <= df_knn_matched['evdt_cohort_start'] + pd.Timedelta(days=365))].shape


In [ ]:
df_knn_matched[(df_knn_matched[col_o]==1)&(df_knn_matched['dod'].notnull()) & (df_knn_matched['dod'] > df_knn_matched['evdt_cohort_start']) & (df_knn_matched['dod'] <= df_knn_matched['evdt_cohort_start'] + pd.Timedelta(days=365))].shape


In [ ]:
def report_grouped(df, report_col, grouping_col):
    out_count = df.groupby([grouping_col, report_col], observed=True).size().unstack(fill_value=0)
    out_percentage = out_count.div(out_count.sum(axis=1), axis=0) * 100
    out_stats_combined = out_count.astype(str) + ' (' + out_percentage.round(4).astype(str) + '%)'
    print(out_stats_combined.T)

def report_total(df, report_col):
    out_count = df[report_col].value_counts().sort_index()
    out_percentage = (out_count / out_count.sum()) * 100
    out_stats_combined = out_count.astype(str) + ' (' + out_percentage.round(4).astype(str) + '%)'
    print(out_stats_combined)

In [ ]:
contorl_knn = df_knn_matched[df_knn_matched[col_o]==0]

In [ ]:
contorl_knn.shape

In [ ]:
report_total(contorl_knn, "sex")

In [ ]:
df_knn_matched[col_o].value_counts()/df.shape[0]*100

In [ ]:
print(round(contorl_knn["age_cohort_start"].mean(), 4))
print(round(contorl_knn["age_cohort_start"].std(), 4))

In [ ]:
report_total(contorl_knn, "age_cohort_start_10")

In [ ]:
print(round(contorl_knn["age_asthma"].mean(), 4))
print(round(contorl_knn["age_asthma"].std(), 4))

In [ ]:
print(round(contorl_knn["follow_up_asthma_pre_cohort_start"].mean(), 4))
print(round(contorl_knn["follow_up_asthma_pre_cohort_start"].std(), 4))

In [ ]:
report_total(contorl_knn, "late_onset_asthma_40")

In [ ]:
report_total(contorl_knn, "isin_deaths")

In [ ]:
report_total(contorl_knn, "eth_grouped_1")

In [ ]:
report_total(contorl_knn, "eth_white")

In [ ]:
report_total(contorl_knn, "Anxiety")

In [ ]:
for item in contorl_knn.columns:
    print(item)

In [ ]:
report_total(contorl_knn, "CKD")

In [ ]:
report_total(contorl_knn, "CVD")

In [ ]:
report_total(contorl_knn, "COPD")

In [ ]:
report_total(contorl_knn, "Diabetes")

In [ ]:
report_total(contorl_knn, "Hypertension")

In [ ]:
report_total(contorl_knn, "pheno_dvt_pre_cohort_start")

In [ ]:
report_total(contorl_knn, "pheno_cardiomyopathy_pre_cohort_start")

In [ ]:
report_total(contorl_knn, "pheno_af_pre_cohort_start")

In [ ]:
report_total(contorl_knn, "pheno_ami_pre_cohort_start")

In [ ]:
report_total(contorl_knn, "pheno_hf_pre_cohort_start")

In [ ]:
report_total(contorl_knn, "pheno_pad_pre_cohort_start")

In [ ]:
report_total(contorl_knn, "pheno_pe_pre_cohort_start")

In [ ]:
report_total(contorl_knn, "pheno_stroke_pre_cohort_start")

In [ ]:
report_total(contorl_knn, "pheno_depression_pre_cohort_start")

In [ ]:
report_total(contorl_knn, "chest_pain_field")

In [ ]:
report_total(contorl_knn, "shortness_breath_field")

In [ ]:
report_total(contorl_knn, "wheeze_field")

In [ ]:
[x for x in contorl_knn.columns if "Pre"  in x]

In [ ]:
report_total(contorl_knn, "desc_smoking_at_baseline_Previous")

In [ ]:
report_total(contorl_knn, "desc_smoking_at_baseline_Unknown")

In [ ]:
report_total(contorl_knn, "Pre OCS")

In [ ]:
report_total(contorl_knn, "Pre exacerbation")

In [ ]:
report_total(contorl_knn, "BMI≥30")

In [ ]:
report_total(contorl_knn, "Cardinal symptomps")

In [ ]:
report_total(contorl_knn, "Current smoker")

In [ ]:
condition = (contorl_knn['dod'].notnull()) & (contorl_knn['dod'] > contorl_knn['evdt_cohort_start']) & (df['dod'] <= df['evdt_cohort_start'] + pd.Timedelta(days=365))

contorl_knn[condition].shape

In [ ]:
contorl_knn[condition].shape[0]/contorl_knn.shape[0]*100

In [ ]:
def divide(a, b):
    print(a/b*100)
    

In [ ]:
divide(18, 2713)